<a href="https://colab.research.google.com/github/anushah-200/SATARK_AI/blob/main/notebooks/03_Fault_Injection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
import os
import pandas as pd
import numpy as np

PROJECT_PATH = "/content/drive/MyDrive/SATARK_AI"

RAW_PATH = f"{PROJECT_PATH}/data/raw"
PROCESSED_PATH = f"{PROJECT_PATH}/data/processed"
SYNTHETIC_PATH = f"{PROJECT_PATH}/data/synthetic"

os.makedirs(SYNTHETIC_PATH, exist_ok=True)

print("Project path:", PROJECT_PATH)

Project path: /content/drive/MyDrive/SATARK_AI


In [36]:
processed_file = f"{PROCESSED_PATH}/processed_meteostat_data.csv"

data = pd.read_csv(processed_file)

data["timestamp"] = pd.to_datetime(data["timestamp"])

print("Shape:", data.shape)
data.head()

Shape: (43100, 28)


,timestamp,station_id,station_name,latitude,longitude,temperature,humidity,pressure,wind_speed,wind_direction,...,temperature_deviation_24h,temperature_zscore_24h,pressure_rolling_mean_24h,pressure_rolling_std_24h,pressure_deviation_24h,pressure_zscore_24h,humidity_rolling_mean_24h,humidity_rolling_std_24h,humidity_deviation_24h,humidity_zscore_24h
0,2025-01-01 00:00:00,42131,Hissar,29.1667,75.7333,5.4,97.0,1020.7,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 01:00:00,42131,Hissar,29.1667,75.7333,6.1,98.0,1019.5,5.0,283.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-01 02:00:00,42131,Hissar,29.1667,75.7333,6.1,94.0,1019.8,5.4,274.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-01 03:00:00,42131,Hissar,29.1667,75.7333,8.0,97.0,1021.5,0.0,0.0,...,2.133333,5.278631,1020.000,0.624500,1.500,2.401922,96.333333,2.081666,0.666667,0.320256
4,2025-01-01 04:00:00,42131,Hissar,29.1667,75.7333,7.6,96.0,1020.8,6.8,276.0,...,1.200000,1.074747,1020.375,0.906918,0.425,0.468620,96.500000,1.732051,-0.500000,-0.288675


In [37]:
synthetic_data = data.copy()

In [38]:
for column in ["temperature", "pressure", "humidity"]:
    synthetic_data[f"original_{column}"] = synthetic_data[column]

In [39]:
synthetic_data["is_anomaly"] = 0
synthetic_data["fault_type"] = "normal"
synthetic_data["fault_severity"] = "none"

In [40]:
fault_types = [
    "spike",
    "drop",
    "frozen",
    "drift",
    "missing",
    "multivariate"
]

variables = [
    "temperature",
    "pressure",
    "humidity"
]

severity_levels = [
    "mild",
    "moderate",
    "severe"
]

In [41]:
severity_magnitude = {
    "temperature": {
        "mild": 5,
        "moderate": 10,
        "severe": 20
    },

    "pressure": {
        "mild": 5,
        "moderate": 10,
        "severe": 20
    },

    "humidity": {
        "mild": 10,
        "moderate": 25,
        "severe": 40
    }
}

In [42]:
injected_regions = []
def overlaps_existing_fault(station_id, start_time, end_time):

    for region in injected_regions:

        if region["station_id"] != station_id:
            continue

        if start_time < region["end_time"] and end_time > region["start_time"]:
            return True

    return False

def register_fault(station_id, start_time, end_time):

    injected_regions.append({
        "station_id": station_id,
        "start_time": start_time,
        "end_time": end_time
    })

In [43]:
def inject_spike(
    df,
    station_id,
    start_time,
    duration,
    variable,
    severity
):

    end_time = start_time + pd.Timedelta(hours=duration)

    mask = (
        (df["station_id"] == station_id) &
        (df["timestamp"] >= start_time) &
        (df["timestamp"] < end_time)
    )

    magnitude = severity_magnitude[variable][severity]

    df.loc[mask, variable] += magnitude

    df.loc[mask, "is_anomaly"] = 1
    df.loc[mask, "fault_type"] = f"{variable}_spike"
    df.loc[mask, "fault_severity"] = severity

    return df

In [44]:
def inject_drop(
    df,
    station_id,
    start_time,
    duration,
    variable,
    severity
):

    end_time = start_time + pd.Timedelta(hours=duration)

    mask = (
        (df["station_id"] == station_id) &
        (df["timestamp"] >= start_time) &
        (df["timestamp"] < end_time)
    )

    magnitude = severity_magnitude[variable][severity]

    df.loc[mask, variable] -= magnitude

    df.loc[mask, "is_anomaly"] = 1
    df.loc[mask, "fault_type"] = f"{variable}_drop"
    df.loc[mask, "fault_severity"] = severity

    return df

In [45]:
def inject_frozen(
    df,
    station_id,
    start_time,
    duration,
    variable,
    severity
):

    end_time = start_time + pd.Timedelta(hours=duration)

    mask = (
        (df["station_id"] == station_id) &
        (df["timestamp"] >= start_time) &
        (df["timestamp"] < end_time)
    )

    indices = df.index[mask]

    if len(indices) == 0:
        return df

    frozen_value = df.loc[indices[0], variable]

    df.loc[indices, variable] = frozen_value

    df.loc[indices, "is_anomaly"] = 1
    df.loc[indices, "fault_type"] = f"{variable}_frozen"
    df.loc[indices, "fault_severity"] = severity

    return df

In [46]:
def inject_drift(
    df,
    station_id,
    start_time,
    duration,
    variable,
    severity
):

    end_time = start_time + pd.Timedelta(hours=duration)

    mask = (
        (df["station_id"] == station_id) &
        (df["timestamp"] >= start_time) &
        (df["timestamp"] < end_time)
    )

    indices = df.index[mask]

    if len(indices) == 0:
        return df

    base_magnitude = {
        "mild": 0.2,
        "moderate": 0.5,
        "severe": 1.0
    }

    drift = base_magnitude[severity]

    for i, idx in enumerate(indices):
        df.loc[idx, variable] += (i + 1) * drift

    df.loc[indices, "is_anomaly"] = 1
    df.loc[indices, "fault_type"] = f"{variable}_drift"
    df.loc[indices, "fault_severity"] = severity

    return df

In [47]:
def inject_missing(
    df,
    station_id,
    start_time,
    duration,
    variable,
    severity
):

    end_time = start_time + pd.Timedelta(hours=duration)

    mask = (
        (df["station_id"] == station_id) &
        (df["timestamp"] >= start_time) &
        (df["timestamp"] < end_time)
    )

    df.loc[mask, variable] = np.nan

    df.loc[mask, "is_anomaly"] = 1
    df.loc[mask, "fault_type"] = f"{variable}_missing"
    df.loc[mask, "fault_severity"] = severity

    return df

In [48]:
def inject_multivariate(
    df,
    station_id,
    start_time,
    duration,
    severity
):

    end_time = start_time + pd.Timedelta(hours=duration)

    mask = (
        (df["station_id"] == station_id) &
        (df["timestamp"] >= start_time) &
        (df["timestamp"] < end_time)
    )

    magnitude = {
        "mild": 5,
        "moderate": 10,
        "severe": 15
    }

    df.loc[mask, "temperature"] += magnitude[severity]

    df.loc[mask, "is_anomaly"] = 1
    df.loc[mask, "fault_type"] = "multivariate_inconsistency"
    df.loc[mask, "fault_severity"] = severity

    return df

In [50]:
test_data = data.copy()

for column in ["temperature", "pressure", "humidity"]:
    test_data[f"original_{column}"] = test_data[column]

test_data["is_anomaly"] = 0
test_data["fault_type"] = "normal"
test_data["fault_severity"] = "none"

In [51]:
test_data = inject_spike(
    test_data,
    station_id="42182",
    start_time=pd.Timestamp("2025-06-15 14:00:00"),
    duration=2,
    variable="temperature",
    severity="severe"
)

In [52]:
test_data[
    (test_data["station_id"] == "42182") &
    (test_data["timestamp"] >= "2025-06-15 12:00:00") &
    (test_data["timestamp"] <= "2025-06-15 17:00:00")
][
    [
        "timestamp",
        "original_temperature",
        "temperature",
        "is_anomaly",
        "fault_type",
        "fault_severity"
    ]
]

,timestamp,original_temperature,temperature,is_anomaly,fault_type,fault_severity


In [53]:
np.random.seed(42)

synthetic_data = data.copy()

for column in ["temperature", "pressure", "humidity"]:
    synthetic_data[f"original_{column}"] = synthetic_data[column]

synthetic_data["is_anomaly"] = 0
synthetic_data["fault_type"] = "normal"
synthetic_data["fault_severity"] = "none"

injected_regions = []

In [54]:
number_of_faults = 100

In [55]:
stations = synthetic_data["station_id"].unique()

successful_faults = 0
attempts = 0
max_attempts = 1000

In [56]:
while successful_faults < number_of_faults and attempts < max_attempts:

    attempts += 1

    station_id = np.random.choice(stations)
    fault_type = np.random.choice(fault_types)
    severity = np.random.choice(severity_levels)

    station_rows = synthetic_data[
        synthetic_data["station_id"] == station_id
    ]

    start_time = np.random.choice(
        station_rows["timestamp"].values
    )

    start_time = pd.Timestamp(start_time)

    duration = np.random.choice([1, 2, 3, 4, 6])

    end_time = start_time + pd.Timedelta(hours=duration)

    if overlaps_existing_fault(
        station_id,
        start_time,
        end_time
    ):
        continue

    variable = np.random.choice(variables)

    if fault_type == "spike":

        synthetic_data = inject_spike(
            synthetic_data,
            station_id,
            start_time,
            duration,
            variable,
            severity
        )

    elif fault_type == "drop":

        synthetic_data = inject_drop(
            synthetic_data,
            station_id,
            start_time,
            duration,
            variable,
            severity
        )

    elif fault_type == "frozen":

        synthetic_data = inject_frozen(
            synthetic_data,
            station_id,
            start_time,
            duration,
            variable,
            severity
        )

    elif fault_type == "drift":

        synthetic_data = inject_drift(
            synthetic_data,
            station_id,
            start_time,
            duration,
            variable,
            severity
        )

    elif fault_type == "missing":

        synthetic_data = inject_missing(
            synthetic_data,
            station_id,
            start_time,
            duration,
            variable,
            severity
        )

    elif fault_type == "multivariate":

        synthetic_data = inject_multivariate(
            synthetic_data,
            station_id,
            start_time,
            duration,
            severity
        )

    register_fault(
        station_id,
        start_time,
        end_time
    )

    successful_faults += 1

print("Faults generated:", successful_faults)
print("Attempts:", attempts)

Faults generated: 100
Attempts: 101


In [57]:
synthetic_data["fault_type"].value_counts()

,count
fault_type,
normal,42778
multivariate_inconsistency,60
temperature_missing,26
pressure_missing,26
pressure_spike,25
pressure_drift,24
temperature_frozen,22
humidity_drift,22
humidity_frozen,21


In [58]:
synthetic_data["fault_severity"].value_counts()

,count
fault_severity,
none,42778
moderate,113
mild,106
severe,103


In [59]:
synthetic_data["is_anomaly"].value_counts()

,count
is_anomaly,
0,42778
1,322


In [60]:
fault_summary = (
    synthetic_data[
        synthetic_data["is_anomaly"] == 1
    ]
    .groupby(["fault_type", "fault_severity"])
    .size()
    .reset_index(name="count")
)

fault_summary

,fault_type,fault_severity,count
0,humidity_drift,mild,14
1,humidity_drift,moderate,3
2,humidity_drift,severe,5
3,humidity_drop,mild,6
4,humidity_drop,moderate,6
5,humidity_frozen,moderate,7
6,humidity_frozen,severe,14
7,humidity_missing,mild,6
8,humidity_missing,severe,3
9,humidity_spike,mild,6


In [61]:
synthetic_data[
    synthetic_data["is_anomaly"] == 1
][
    [
        "timestamp",
        "station_id",
        "original_temperature",
        "temperature",
        "original_pressure",
        "pressure",
        "original_humidity",
        "humidity",
        "fault_type",
        "fault_severity"
    ]
].head(20)

,timestamp,station_id,original_temperature,temperature,original_pressure,pressure,original_humidity,humidity,fault_type,fault_severity
512,2025-01-22 08:00:00,42131,20.5,20.5,1013.7,1003.7,63.0,63.0,pressure_drop,moderate
513,2025-01-22 09:00:00,42131,22.8,22.8,1013.2,1003.2,43.0,43.0,pressure_drop,moderate
1069,2025-02-14 13:00:00,42131,18.0,18.0,1011.6,1012.1,71.0,71.0,pressure_drift,moderate
1070,2025-02-14 14:00:00,42131,17.7,17.7,1011.8,1012.8,67.0,67.0,pressure_drift,moderate
1071,2025-02-14 15:00:00,42131,16.2,16.2,1014.0,1015.5,68.0,68.0,pressure_drift,moderate
1072,2025-02-14 16:00:00,42131,16.4,16.4,1012.2,1014.2,63.0,63.0,pressure_drift,moderate
1073,2025-02-14 17:00:00,42131,15.1,15.1,1012.1,1014.6,65.0,65.0,pressure_drift,moderate
1074,2025-02-14 18:00:00,42131,14.8,14.8,1013.3,1016.3,70.0,70.0,pressure_drift,moderate
1816,2025-03-22 05:00:00,42131,26.0,26.0,1014.5,1014.5,41.0,41.0,humidity_frozen,moderate
1817,2025-03-22 06:00:00,42131,30.2,30.2,1015.5,1015.5,41.0,41.0,humidity_frozen,moderate


In [62]:
synthetic_file = (
    f"{SYNTHETIC_PATH}/synthetic_fault_data.csv"
)

synthetic_data.to_csv(
    synthetic_file,
    index=False
)

print("Saved:")
print(synthetic_file)

Saved:
/content/drive/MyDrive/SATARK_AI/data/synthetic/synthetic_fault_data.csv
